# DNN — Edge-Based Strategy Discovery

**Method:** Train model on first 80% of candles, generate predictions on unseen 20%, then run edge-based grid search (walk-forward 5 folds on validation set only).

**Edge = max(prob, 1-prob) - ask_price.** Only bet when the model's confidence exceeds the market price.

**Output:** `data/optimal_strategy_dnn.json`

In [1]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json
import os
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from strategy_engine import StrategyGrid, WalkForwardEvaluator

sys.path.insert(0, "../..")

np.random.seed(42)
MAX_BID = 0.85
_is_dnn = False

## 1. Load Data & Split

In [2]:
# Load feature config
with open("../../data/optimal_features_dnn.json") as f:
    feat_config = json.load(f)
feat_cols = feat_config["features"]

# Load all snapshots
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# 80/20 split by candle (time-ordered)
candle_ids = df["candle_id"].unique()
split = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split])
val_ids = set(candle_ids[split:])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]
print(f"Train: {len(train_ids)} candles ({len(df_train):,} snapshots)")
print(f"Val:   {len(val_ids)} candles ({len(df_val):,} snapshots)")

Train: 5208 candles (246,168 snapshots)
Val:   1302 candles (60,452 snapshots)


## 2. Train Model on 80% Only

In [3]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feat_cols].fillna(0).values)
y_train = df_train["target"].values

# DNN uses exported model (can't easily retrain in strategy notebook)
# This is a known limitation — DNN strategy eval uses full-data model
model = torch.load("../../models/dnn_v1.pt", weights_only=False)
model.eval()
calibrator_path = "../../models/dnn_calibrator_v1.joblib"
calibrator = joblib.load(calibrator_path) if os.path.exists(calibrator_path) else None
_is_dnn = True
print(f"Model trained on {len(train_ids)} candles")

Model trained on 5208 candles


## 3. Generate Predictions on Unseen 20%

In [4]:
candle_data = []
for cid in df_val["candle_id"].unique():
    snap_rows = df_val[df_val["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue
    truth = int(snap_rows["target"].iloc[0])
    X_val_scaled = scaler.transform(snap_rows[feat_cols].fillna(0).values)
    X_tensor = torch.tensor(X_val_scaled.astype(np.float32), dtype=torch.float32)
with torch.no_grad():
    logits = model(X_tensor)
    probs = torch.sigmoid(logits).numpy().flatten()
if calibrator is not None:
    probs = calibrator.predict(probs)

    snapshots = []
    for i in range(len(snap_rows)):
        ua = snap_rows.iloc[i]["up_best_ask"]
        da = snap_rows.iloc[i]["down_best_ask"]
        snapshots.append(
            {
                "tick": i,
                "elapsed_pct": float(snap_rows.iloc[i]["elapsed_pct"]),
                "pred": int(probs[i] >= 0.5),
                "prob": float(probs[i]),
                "up_ask": float(ua) if ua is not None and np.isfinite(ua) else None,
                "down_ask": float(da) if da is not None and np.isfinite(da) else None,
            }
        )
    candle_data.append({"candle_id": cid, "truth": truth, "snapshots": snapshots})

print(f"Built {len(candle_data)} validation candles (model has NEVER seen these)")

Built 1 validation candles (model has NEVER seen these)


## 4. Edge-Based Grid Search

In [5]:
grid = StrategyGrid()
strategies = grid.generate()
print(f"Grid: {len(strategies)} combinations")

evaluator = WalkForwardEvaluator(strategies=strategies, candle_data=candle_data, n_folds=5, max_bid=MAX_BID)
results_df = evaluator.run()

print("\nTop 5 by Sharpe:")
print(
    results_df[["strategy", "sharpe_mean", "win_rate_mean", "return_mean", "total_bets_mean"]]
    .head(5)
    .to_string(index=False)
)

Grid: 2 combinations

Top 5 by Sharpe:
     strategy  sharpe_mean  win_rate_mean  return_mean  total_bets_mean
edge>=0.05 x1          0.0            0.0         -0.2              0.2
edge>=0.05 x2          0.0            0.0         -0.4              0.4


## 5. Export Best Strategy

In [6]:
best = results_df.iloc[0]
best_cfg = best["_config"]

config = {
    "model": "dnn",
    "strategy": best["strategy"],
    "min_edge": best_cfg.min_edge,
    "max_entries": best_cfg.max_entries,
    "sharpe_mean": round(best["sharpe_mean"], 4),
    "sharpe_std": round(best["sharpe_std"], 4),
    "win_rate": round(best["win_rate_mean"], 4),
    "return_pct": round(best["return_mean"], 2),
    "max_drawdown": round(best["max_dd_mean"], 4),
    "n_folds": 5,
    "grid_size": len(strategies),
    "eval_candles": len(candle_data),
    "eval_method": "walk_forward_5_folds_on_validation_split",
    "total_bets": int(best["total_bets_mean"]),
    "created_at": datetime.now(UTC).isoformat(),
}

out_path = Path("../../data/optimal_strategy_dnn.json")
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Best: min_edge={best_cfg.min_edge}, max_entries={best_cfg.max_entries}")
print(f"Sharpe={best['sharpe_mean']:.4f}, WR={best['win_rate_mean'] * 100:.1f}%, Return={best['return_mean']:+.1f}%")
print(f"\nSaved to {out_path}")

Best: min_edge=0.05, max_entries=1
Sharpe=0.0000, WR=0.0%, Return=-0.2%

Saved to ../../data/optimal_strategy_dnn.json
